<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Memory Management: Stateless vs Stateful AI

Welcome! By the end of this session you will know how to build a chatbot that *remembers* what you said earlier -- and understand exactly why that does not happen by default.


### What you will learn today
- Why Claude forgets between conversations (and why that is intentional)
- How to keep a conversation history so Claude remembers
- How to handle conversations that grow very long
---

---

## Section 1 : The Memory Problem: Why Claude Forgets

### Key Idea

Every time you call Claude's API, it is like **meeting Claude for the first time**.

Claude does **not** store anything from your last conversation. When you close the chat and come back, Claude has no idea who you are or what you talked about.

> **This is intentional for privacy and security.** Your conversations are not mixed with other people's conversations.

### Real-Life Analogy

Imagine a shop employee who has a **perfect memory while you are shopping**, but the moment you leave the store their memory resets completely. The next time you walk in, they greet you like a stranger.

That is Claude. Great memory *inside* one conversation. Zero memory when you come back.

### Connecting to What You Already Know

From **Weekend 1**: Claude reads *tokens*. It processes them one request at a time.

> **New idea:** Claude reads the tokens YOU send. If you do not send the old messages again, Claude never sees them.

### What Really Happens

| Situation | What You Send | What Claude Sees | Result |
|---|---|---|---|
| Ask "My name is Alice" | New message only | "My name is Alice" | Claude says "Nice to meet you!" |
| **New** chat: "What is my name?" | New message only | "What is my name?" | Claude says "I don't know" |
| **Same** chat: "What is my name?" | Old + new message | Both messages | Claude says "Alice" |

The difference? In the same chat, your app sends the old messages too. In a new chat, they are gone.

### Quick Check

> **Question:** If you ask Claude 'What is your favourite food?' right now in a brand-new chat, what will Claude say? Why?

> **Answer:** Claude will say it does not know, because this is the first message in this conversation. Claude has never been told your favourite food.

---

In [2]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 20.7 MB/s eta 0:00:00


In [5]:
from anthropic import Anthropic
from google.colab import userdata
key = userdata.get('MY_API_KEY')

client = Anthropic(api_key=key)

In [6]:
# DEMO: Prove Claude Forgets
# We make TWO separate API calls.
# Each call sends only ONE message -- no history.
# Watch Claude forget between calls.

# --- Call 1: Tell Claude your name ---
print('--- Call 1: Telling Claude your name ---')

response_1 = client.messages.create(
    model='claude-haiku-4-5-20251001',     # Fast model -- great for learning
    max_tokens=100,               # Limit response length to save cost
    messages=[
        {'role': 'user', 'content': 'My name is Arul. Nice to meet you!'}
    ]
)

# Extract the text from Claude's response
reply_1 = response_1.content[0].text  # content[0] is the first response block
print(f'Claude said: {reply_1}')

print()
print('--- Call 2: Ask Claude your name (NEW call, no history) ---')

response_2 = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=100,
    messages=[
        {'role': 'user', 'content': 'What is my name?'}  # Only the new message -- no history!
    ]
)

reply_2 = response_2.content[0].text  # Get the text of the reply
print(f'Claude said: {reply_2}')

print()
print('--- Observation ---')
print('Call 1 and Call 2 are completely separate.')
print('Claude never received the first message in Call 2.')
print('So Claude could not know your name. This is STATELESS behaviour.')


--- Call 1: Telling Claude your name ---
Claude said: Nice to meet you too, Arul! 👋 

Feel free to ask me anything or chat about whatever's on your mind. What brings you here today?

--- Call 2: Ask Claude your name (NEW call, no history) ---
Claude said: I don't have any information about your name. Each conversation with me starts fresh without access to previous conversations or personal details about you.

If you'd like to tell me your name, feel free to share!

--- Observation ---
Call 1 and Call 2 are completely separate.
Claude never received the first message in Call 2.
So Claude could not know your name. This is STATELESS behaviour.


---

## Section 2 : How to Make Claude Remember: The Conversation List

### The Solution

Keep a **list** of every message -- both what you said AND what Claude said.

Every time you send a new message, send the **entire list** plus the new message.

Claude reads them all and 'remembers' the full conversation.

### Stateless vs Stateful -- Side by Side

**Stateless (Claude forgets):**
```
Request 1: "Hi, my name is Bob"
Claude: "Nice to meet you"
[Everything forgotten]

Request 2: "Do you remember me?"
Claude: "No, this is our first chat"
```

**Stateful (Claude remembers):**
```
Request 1: "Hi, my name is Bob"
Claude: "Nice to meet you"
[Your app saves both messages in a list]

Request 2: [old message 1, old response 1, NEW message 2]
Claude: "Yes! You are Bob!"
```

The only difference is that **your code saves and re-sends the history**.

### Message Format

Each message in the list is a small dictionary with two keys:

```json
{
    "role": "user",
    "content": "What is 2 + 2?"
}
```

- `role` is either `"user"` (you) or `"assistant"` (Claude)
- `content` is the text of the message

### A Real Conversation List

```json
[
    {"role": "user",      "content": "My name is Sarah"},
    {"role": "assistant", "content": "Nice to meet you, Sarah!"},
    {"role": "user",      "content": "What is my name?"},
    {"role": "assistant", "content": "Your name is Sarah!"}
]
```

Notice how the list grows with every turn. You send the **whole list** each time.

### Connecting to What You Already Know

In **Weekend 3** you defined tool descriptions as dictionaries. This is the same idea -- you are describing the structure of a conversation using dictionaries.

### Simple Rule to Remember

> To make Claude remember: **send the entire list every time**.

### Quick Check

> **Question:** If you have 5 previous messages and want to ask a 6th question, how many messages does Claude need to receive in total?

> **Answer:** 6 -- the 5 old ones plus the 1 new one.

---

In [7]:
# DEMO: Stateful Call Using a Manual List
# We manually build the conversation list and prove Claude remembers.

print('--- Building a conversation list by hand ---')
print()

# Start with an empty list -- no history yet
conversation_history = []  # This will grow with every message

# Turn 1
user_message_1 = 'My name is Bob. Nice to meet you!'

# Add the user message to the list before calling Claude
conversation_history.append({
    'role': 'user',
    'content': user_message_1
})

# Send the full list to Claude
response_turn_1 = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=100,
    messages=conversation_history  # Send the FULL list
)

claude_reply_1 = response_turn_1.content[0].text  # Extract text

# Save Claude's reply to the list too
conversation_history.append({
    'role': 'assistant',
    'content': claude_reply_1
})

print(f'Turn 1 -- You: {user_message_1}')
print(f'Turn 1 -- Claude: {claude_reply_1}')
print(f'History size: {len(conversation_history)} messages')
print()

# Turn 2: Ask Claude to recall the name
user_message_2 = 'What is my name?'

# Add new user message to the list
conversation_history.append({
    'role': 'user',
    'content': user_message_2
})

# Send the full list -- now it has 3 messages (2 old + 1 new)
response_turn_2 = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=100,
    messages=conversation_history  # Send ALL history!
)

claude_reply_2 = response_turn_2.content[0].text

# Save Claude's reply
conversation_history.append({
    'role': 'assistant',
    'content': claude_reply_2
})

print(f'Turn 2 -- You: {user_message_2}')
print(f'Turn 2 -- Claude: {claude_reply_2}')
print(f'History size: {len(conversation_history)} messages')
print()
print('Claude remembered the name! This is STATEFUL behaviour.')
print('The secret: we re-sent all old messages in Turn 2.')


--- Building a conversation list by hand ---

Turn 1 -- You: My name is Bob. Nice to meet you!
Turn 1 -- Claude: Nice to meet you too, Bob! I'm Claude, an AI assistant. Feel free to ask me anything or let me know how I can help you today.
History size: 2 messages

Turn 2 -- You: What is my name?
Turn 2 -- Claude: Your name is Bob! You told me that at the start of our conversation.
History size: 4 messages

Claude remembered the name! This is STATEFUL behaviour.
The secret: we re-sent all old messages in Turn 2.


---

## Section 3 : Building Your First Stateful Chatbot

### Goal

We are going to package everything from Section 2 into a clean, reusable class called `SimpleChatMemory`.

Instead of managing the list by hand every time, the class does it for us.

### Connecting to What You Already Know

You have already called Claude's API (Weekends 2 and 3). We are just adding one new piece: a memory object that grows as you chat.

---

In [8]:
# Class: SimpleChatMemory
# Stores all messages in the conversation.
# Use it to add messages, retrieve them, and print the conversation.

class SimpleChatMemory:

    def __init__(self):
        # Start with an empty list -- no history yet
        self.messages = []  # This list grows as the conversation continues

    def add_user_message(self, text):
        # Build a user message dictionary
        new_message = {
            'role': 'user',  # Mark this as coming from the user
            'content': text  # Store the text they typed
        }
        self.messages.append(new_message)  # Add to the end of the list

    def add_assistant_message(self, text):
        # Build an assistant message dictionary
        new_message = {
            'role': 'assistant',  # Mark this as coming from Claude
            'content': text       # Store the text Claude replied
        }
        self.messages.append(new_message)  # Add to the end of the list

    def get_all_messages(self):
        # Return the full conversation list -- this is what we send to Claude
        return self.messages  # Hand back the complete list

    def show_conversation(self):
        # Print the conversation in a readable format
        print('=== Conversation History ===')
        for message in self.messages:     # Loop through every message
            role = message['role']        # Get who said it
            content = message['content']  # Get what was said
            print(f'  {role}: {content}') # Print in readable format
        print(f'=== Total: {len(self.messages)} messages ===')

    def clear(self):
        # Wipe the conversation history completely
        self.messages = []  # Replace list with a new empty list
        print('Conversation cleared.')


# Quick test to confirm the class works
print('--- Testing SimpleChatMemory ---')
test_memory = SimpleChatMemory()               # Create a memory object
test_memory.add_user_message('Hello!')         # Add a user message
test_memory.add_assistant_message('Hi!')       # Add an assistant message
test_memory.add_user_message('How are you?')   # Add another user message
test_memory.show_conversation()                # Print everything


--- Testing SimpleChatMemory ---
=== Conversation History ===
  user: Hello!
  assistant: Hi!
  user: How are you?
=== Total: 3 messages ===


### The chat_with_memory Function

Now let us write a function that:
1. Adds your new message to the memory
2. Sends the **full history** to Claude
3. Gets Claude's reply
4. Saves the reply to memory
5. Shows you the current conversation

---

In [9]:
# Function: chat_with_memory
# Combines the API call with memory management in one place.

def chat_with_memory(user_input, memory_object, api_client):
    # Parameters:
    #   user_input    - The new message you want to send (a string)
    #   memory_object - A SimpleChatMemory instance that stores the history
    #   api_client    - The Anthropic client created in the Setup cell

    print(f"--- You said: '{user_input}' ---")

    # Step 1: Add the new user message to the history
    memory_object.add_user_message(user_input)   # Save it before sending

    # Step 2: Send ALL messages (full history) to Claude
    response = api_client.messages.create(
        model='claude-haiku-4-5-20251001',                     # Fast, free-tier-friendly
        max_tokens=300,                               # Allow up to 300 tokens
        messages=memory_object.get_all_messages()     # Send the complete history!
    )

    # Step 3: Extract the text from Claude's response
    claude_response = response.content[0].text   # Get the first block's text

    # Step 4: Save Claude's response to memory
    memory_object.add_assistant_message(claude_response)  # Store for next time

    # Step 5: Print Claude's reply
    print(f'Claude said: {claude_response}')
    print()

    return claude_response  # Return it in case you need it in code


# Live Example: Watch Memory Grow
print('=== Starting a stateful chat ===')
print()

my_memory = SimpleChatMemory()  # Create a fresh memory object

# Turn 1 -- introduce yourself
chat_with_memory('My name is Alex. I like Python.', my_memory, client)

# Turn 2 -- test if Claude remembers
chat_with_memory('What is my name?', my_memory, client)

# Turn 3 -- go further
chat_with_memory('Tell me a short joke related to my interests.', my_memory, client)

# Print the full conversation at the end
print('--- Full conversation stored in memory: ---')
my_memory.show_conversation()


=== Starting a stateful chat ===

--- You said: 'My name is Alex. I like Python.' ---
Claude said: Nice to meet you, Alex! Python is a great choice. It's known for being readable, versatile, and beginner-friendly while still being powerful enough for complex projects.

Is there anything Python-related you'd like to discuss or get help with? Whether it's learning, debugging, project ideas, or best practices, I'm happy to help!

--- You said: 'What is my name?' ---
Claude said: Your name is Alex.

--- You said: 'Tell me a short joke related to my interests.' ---
Claude said: Here's one for you:

Why do Python programmers make terrible dancers?

Because they keep getting stuck in infinite loops! 🐍

--- Full conversation stored in memory: ---
=== Conversation History ===
  user: My name is Alex. I like Python.
  assistant: Nice to meet you, Alex! Python is a great choice. It's known for being readable, versatile, and beginner-friendly while still being powerful enough for complex projects.

### Quick Check

> **Question:** Where is the conversation history stored?

> **Answer:** In YOUR code -- in the `SimpleChatMemory` object. Not inside Claude. You manage it.

---

---

## Section 4 : What Happens When Conversations Get Too Long?

### Key Concept: The Context Window

Claude can only read a certain amount of text in a single request. That limit is called the **context window**.

**Claude's context window: 200,000 tokens** (on standard plans)

### What Is a Token?

From Weekend 1, you know Claude counts tokens, not words or characters.

A rough guide (for learning -- not exact):

```
Quick Estimates:

"What is 2+2?"              ~  4 tokens
A typical chat message      ~  50-100 tokens
One page of a book          ~  500 tokens
Claude's total window       ~  200,000 tokens
                               (like ~400 pages of a book)

Rough maths:
200,000 / 100 = 2,000 messages before hitting the limit
(if each message is ~100 tokens)
```

> **Disclaimer:** These are rough estimates for learning. For exact counts, use the [Token Counting API](https://docs.claude.com/en/docs/build-with-claude/token-counting).

### Why Does This Matter?

If you chat with Claude for a long time, the message list grows. Eventually the total tokens across all messages will approach 200,000, and then Claude cannot fit them all in one request.

### The Solution: Sliding Window

Keep only the **most recent messages**. Drop the oldest ones when you get too many.

```
Before trimming:
[old msg 1, old msg 2, ..., recent msg 1998, recent msg 1999, recent msg 2000]

After trimming (keep last 1500):
[recent msg 501, ..., recent msg 1998, recent msg 1999, recent msg 2000]
```

You lose some history, but Claude can still help with the current topic.

### Visual Token Budget

```
Token budget:  [==================] 200,000 available

Small chat:    [##                ]  ~20,000 used  -- plenty of room
Growing chat:  [############      ] ~150,000 used  -- getting full
Full chat:     [##################] ~200,000 used  -- NEED TO TRIM!

When full -> remove oldest messages and keep going.
```

### Quick Check

> **Question:** If Claude's window is 200,000 tokens and each message is about 100 tokens, roughly how many messages can fit?

> **Answer:** 200,000 / 100 = 2,000 messages

---

In [10]:
# Simple Token Estimator
# Real token counting requires an API call (free but needs a network).
# For this lesson we use a rough character-based estimate.
# Anthropic's own approximation: 1 token ~= 3.5 English characters.

def count_tokens_approximate(text):
    # Count the characters in the string
    characters = len(text)
    # Divide by 3.5 -- Anthropic's approximation for English text
    approximate_tokens = characters / 3.5
    # Round down to a whole number
    return int(approximate_tokens)

    # DISCLAIMER: This is a rough estimate for learning only.
    # For production code, use the real Token Counting API:
    # https://docs.claude.com/en/docs/build-with-claude/token-counting


# Test the estimator on a few sample strings
print('--- Token Estimation Examples ---')
print()

sample_messages = [
    'Hello!',
    'What is 2 + 2?',
    'My name is Alice and I am learning how to build stateful chatbots with Claude.',
    'A' * 500   # A string of 500 characters to show the scale
]

for sample in sample_messages:   # Loop through each sample
    token_count = count_tokens_approximate(sample)   # Estimate tokens
    display_text = sample[:60]                       # Show only first 60 chars
    if len(sample) > 60:
        display_text = display_text + '...'          # Add '...' if cut
    print(f"  Text: '{display_text}'")
    print(f'  Characters: {len(sample)}  -->  Estimated tokens: {token_count}')
    print()

print('DISCLAIMER: These are rough estimates for learning.')
print('Use the Token Counting API for accurate production results.')


--- Token Estimation Examples ---

  Text: 'Hello!'
  Characters: 6  -->  Estimated tokens: 1

  Text: 'What is 2 + 2?'
  Characters: 14  -->  Estimated tokens: 4

  Text: 'My name is Alice and I am learning how to build stateful cha...'
  Characters: 78  -->  Estimated tokens: 22

  Text: 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...'
  Characters: 500  -->  Estimated tokens: 142

DISCLAIMER: These are rough estimates for learning.
Use the Token Counting API for accurate production results.


### The Sliding Window Function

Now let us write the function that removes old messages when the list gets too long.

---

In [11]:
# Sliding Window: Trim Old Messages
# When a conversation gets too long, we remove the OLDEST messages first.
# This keeps the most recent context and frees up space.

def apply_sliding_window(message_list, max_messages_to_keep):
    # Count how many messages we have right now
    current_count = len(message_list)

    if current_count > max_messages_to_keep:
        # We have too many messages -- need to trim
        messages_to_remove = current_count - max_messages_to_keep  # How many to drop
        trimmed_messages = message_list[messages_to_remove:]        # Keep only recent ones

        print(f'Conversation too long! Had {current_count} messages.')
        print(f'Removed {messages_to_remove} oldest messages.')
        print(f'Keeping {len(trimmed_messages)} most recent messages.')

        return trimmed_messages  # Return the shorter list

    else:
        # We are still within the limit -- no trimming needed
        print(f'Conversation is fine. {current_count} messages (limit: {max_messages_to_keep}).')
        return message_list  # Return the list unchanged


# Test 1: Under the limit
print('--- Test 1: Conversation under the limit ---')
short_conversation = [
    {'role': 'user',      'content': 'Message 1'},
    {'role': 'assistant', 'content': 'Reply 1'},
    {'role': 'user',      'content': 'Message 2'},
    {'role': 'assistant', 'content': 'Reply 2'},
]
result_1 = apply_sliding_window(short_conversation, max_messages_to_keep=10)
print(f'Returned {len(result_1)} messages.')
print()

# Test 2: Over the limit
print('--- Test 2: Conversation over the limit ---')
long_conversation = []  # Build a longer list programmatically
for turn_number in range(1, 11):  # 10 turns = 20 messages
    long_conversation.append({'role': 'user',      'content': f'User message {turn_number}'})
    long_conversation.append({'role': 'assistant', 'content': f'Claude reply {turn_number}'})

result_2 = apply_sliding_window(long_conversation, max_messages_to_keep=6)
print(f'Returned {len(result_2)} messages.')
print(f"First kept message: '{result_2[0]['content']}'")
print(f"Last kept message:  '{result_2[-1]['content']}'")


--- Test 1: Conversation under the limit ---
Conversation is fine. 4 messages (limit: 10).
Returned 4 messages.

--- Test 2: Conversation over the limit ---
Conversation too long! Had 20 messages.
Removed 14 oldest messages.
Keeping 6 most recent messages.
Returned 6 messages.
First kept message: 'User message 8'
Last kept message:  'Claude reply 10'


---

## Summary : What You Learned Today

**Claude is stateless by default** -- every API call is a fresh start; no memory between calls

**You build stateful systems** -- by keeping a list of all messages in your own code

**Message format matters** -- each message needs a `role` and `content` key

**Context window is 200,000 tokens** -- the maximum input size per request on standard plans

**Conversations grow** -- history gets longer with every turn, using more of the context window

**Sliding windows manage long chats** -- remove oldest messages first when you hit the limit

---

### The Simple Rule

```
To make Claude remember:

1. Keep a list of ALL messages (user + assistant)
2. Send the ENTIRE list every time you talk to Claude
3. Claude reads all of them and understands the full context
4. When the list gets too long, remove the OLDEST messages first
```

---

### Where Is the Memory Stored?

> **In YOUR code.** Claude does not store it. You own it, you manage it, you decide when to clear it.

---

### Real-World Uses

| Application | How Memory Helps |
|---|---|
| Customer support chatbot | Remembers the problem described at the start of the conversation |
| Personal assistant | Remembers preferences you mentioned earlier |
| Tutoring system | Remembers what topics a student has already covered |
| Writing assistant | Remembers the style guide you specified at the beginning |

---

### Resources

- **Claude Context Window (200k tokens):** https://support.claude.com/en/articles/11647753-how-do-usage-and-length-limits-work
- **Token Counting API (free to call):** https://docs.claude.com/en/docs/build-with-claude/token-counting
- **Building Chatbots -- Anthropic Docs:** https://docs.anthropic.com/en/docs/build-with-claude/
- **Python Anthropic SDK:** https://github.com/anthropics/anthropic-sdk-python

---

### What Is Next?

Now that you can build stateful chatbots, great next topics are:

- **Summarisation strategies** -- summarise old messages instead of deleting them
- **System prompts** -- give Claude a persistent persona or ruleset at the top of every request
- **Streaming responses** -- show Claude's reply word by word as it generates
- **Multi-user sessions** -- manage separate memory objects for different users

Well done!